In [ ]:
from pathlib import Path

from wekeo_iasi_l3.download import get_IASI_products
from wekeo_iasi_l3.reader_l2.reader import read_iasi_l2
from wekeo_iasi_l3.global_accumulator import GlobalAccumulator2D

import xarray as xr

from wekeo_iasi_l3.hygeos_core import log
from wekeo_iasi_l3.hygeos_core import env
from datetime import date
    
def  get_iasi_day_files(day: date) -> list[Path]:
    """Download and Compile multiple FRP files into a single xarray Dataset."""
    
    log.info(f"Downloading FRP products for date: {day}")
    
    eday = day.strftime("%Y-%m-%d")
    sday = day.strftime("%Y-%m-%d") # same day for single day query

    files = get_IASI_products(
        start_date = sday,
        end_date = eday,
    )
    
    return files

def compute_iasi_l3(files: list[Path], neq: int, variables: list[str]=["INTEGRATED_CO"], remove_night: bool=False) -> xr.Dataset:
    WIDTH = neq

    acc = GlobalAccumulator2D(width = WIDTH, height = int(WIDTH/2))
    ds = xr.Dataset()
    
    bad_files = []
    
    for variable in variables:
        
        log.info(f"Processing variable: {variable}")
        
        for idx, file in enumerate(files):
            if file in bad_files:
                continue
            try:
                log.info(f"Reading file {idx + 1}/{len(files)}: {file} ")
                dataset = read_iasi_l2(file, variables=variables)
                if remove_night:
                    dataset[variable] = dataset[variable].where(abs(dataset.SZA) < 90)
                acc.add(dataset[variable].values, lat=dataset.latitude.values, lon=dataset.longitude.values)
            except Exception as e:
                log.error(f"Error processing file {file}: {e}")
                bad_files.append(file)
                continue
        
        # append the gridded L3 to the output dataset
        ds[variable + "_MEAN"] = acc.get_mean_data_array()
        ds[variable + "_CNT"] =  acc.get_cnt_data_array()
        
        if "unit" in dataset[variable].attrs: 
            ds[variable + "_MEAN"].attrs["unit"] = dataset[variable].attrs["unit"]
            
        if "long_name" in dataset[variable].attrs: 
            ds[variable + "_MEAN"].attrs["long_name"] = dataset[variable].attrs["long_name"] + " gridded mean"

    ds.attrs["description"] = f"Gridded L3 dataset from IASI Level 2 products"
    ds.attrs["source_files"] = ", ".join(f.name for f in files)

    return ds
    
    
# compute the reprojected L3 dataset
PROPER_RESOLUTION_WIDTH = 3272  # ~1km at the equator, adjust as needed
SIMPLER_RESOLUTION_WIDTH = 1000  # for demo purposes

files = get_iasi_day_files(date(2024, 8, 13))

# ds_lvl3 = get_iasi_l3(date(2024, 8, 15))
# files = [Path("../test_input/IASI_SND_02_M01_20240816222952Z_20240817001456Z_N_O_20240816233022Z")]
ds = compute_iasi_l3(files, neq=SIMPLER_RESOLUTION_WIDTH, variables=["INTEGRATED_CO"], remove_night=True)

print(ds)

In [ ]:
# Plot L3 gridded data

from wekeo_iasi_l3.plot_L3_IASI import plot_L3_IASI
from pathlib import Path

colormap = "viridis"
figsize = (14, 7)

figdir = Path("figures")
figdir.mkdir(parents=False, exist_ok=True)
# figdir = None  # Disables figure saving

# Plot INTEGRATED_CO variables
plot_L3_IASI(ds, variable='INTEGRATED_CO_MEAN', cmap='plasma', figsize=figsize, save_fig_dir=figdir)
plot_L3_IASI(ds, variable='INTEGRATED_CO_CNT', cmap='plasma', figsize=figsize, save_fig_dir=figdir)